<a href="https://colab.research.google.com/github/Khamidrees/DSN-ML-Track-Competition-2026-Bootcamp/blob/main/DSN_ML_Track_Submission_18_Notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DSN ML Track Submission 18 Notebook

In [15]:
import os
import numpy as np
import pandas as pd

from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor

!pip install catboost
from catboost import CatBoostRegressor

SEED = 42
BASE = "/content"

TRAIN_PATH = os.path.join(BASE, "train.csv")
TEST_PATH  = os.path.join(BASE, "test.csv")
OUTPUT_PATH = os.path.join(BASE, "submission_18.csv")

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

print("Train:", train.shape)
print("Test :", test.shape)
print(train.columns.tolist())

Train: (6818, 13)
Test : (1705, 12)
['id', 'product_code', 'product_weight_kg', 'fat_content', 'shelf_visibility', 'product_category', 'product_price', 'store_code', 'store_age_years', 'store_size', 'store_location_tier', 'store_format', 'total_sales']


## 1. tree-ensemble model

In [16]:
def clean_category(df):
    df = df.copy()
    df["fat_content"] = df["fat_content"].astype(str).str.strip()
    df["product_category"] = df["product_category"].astype(str).str.strip().str.lower()

    cat_map = {
        "frozen foods": "Frozen Foods",
        "health and hygiene": "Health and Hygiene",
        "canned": "Canned",
        "soft drinks": "Soft Drinks",
        "meat": "Meat",
        "snack foods": "Snack Foods",
        "baking goods": "Baking Goods",
        "household": "Household",
        "dairy": "Dairy",
        "fruits and vegetables": "Fruits and Vegetables",
        "breakfast": "Breakfast",
        "breads": "Breads",
        "others": "Others",
        "seafood": "Seafood",
        "hard drinks": "Hard Drinks",
        "starchy foods": "Starchy Foods",
    }

    df["product_category"] = df["product_category"].map(cat_map)
    return df

train = clean_category(train)
test = clean_category(test)

all_df = pd.concat(
    [train.assign(_src="train"), test.assign(_src="test")],
    ignore_index=True,
    sort=False
)

# Product weight
weight_by_product = all_df.groupby("product_code")["product_weight_kg"].transform("mean")
all_df["product_weight_kg"] = all_df["product_weight_kg"].fillna(weight_by_product)

weight_by_cat = all_df.groupby("product_category")["product_weight_kg"].transform("mean")
all_df["product_weight_kg"] = all_df["product_weight_kg"].fillna(weight_by_cat)

# Store size
store_size_map = (
    all_df.dropna(subset=["store_size"])
    .drop_duplicates("store_code")
    .set_index("store_code")["store_size"]
    .to_dict()
)

missing_stores = set(all_df["store_code"].unique()) - set(store_size_map.keys())

for s in missing_stores:
    fmt = all_df.loc[all_df["store_code"] == s, "store_format"].iloc[0]
    mode = all_df.loc[all_df["store_format"] == fmt, "store_size"].mode()
    store_size_map[s] = mode.iloc[0] if len(mode) else "Medium"

all_df["store_size"] = all_df["store_code"].map(store_size_map)

# Feature engineering
all_df["price_per_kg"] = (
    all_df["product_price"] / all_df["product_weight_kg"]
)

all_df["store_avg_price"] = (
    all_df.groupby("store_code")["product_price"].transform("mean")
)

all_df["cat_avg_price"] = (
    all_df.groupby("product_category")["product_price"].transform("mean")
)

all_df["price_vs_cat_avg"] = (
    all_df["product_price"] - all_df["cat_avg_price"]
)

cat_avg_vis = (
    all_df.groupby("product_category")["shelf_visibility"].transform("mean")
)

all_df["vis_vs_cat_avg"] = (
    all_df["shelf_visibility"] - cat_avg_vis
)

all_df["store_age_bin"] = pd.cut(
    all_df["store_age_years"],
    bins=[0, 25, 30, 40, 50],
    labels=["<=25", "26-30", "31-40", "41+"]
)

train_fe = (
    all_df[all_df["_src"] == "train"]
    .drop(columns="_src")
    .reset_index(drop=True)
)

test_fe = (
    all_df[all_df["_src"] == "test"]
    .drop(columns=["_src", "total_sales"])
    .reset_index(drop=True)
)

target = "total_sales"
drop_cols = ["id", "product_code", "store_code", target]

feature_cols = [c for c in train_fe.columns if c not in drop_cols]

cat_cols = train_fe[feature_cols].select_dtypes(
    include=["object", "category"]
).columns.tolist()

X = train_fe[feature_cols].copy()
X_test = test_fe[feature_cols].copy()
y = train_fe[target].copy()

for c in cat_cols:
    X[c] = X[c].astype(str)
    X_test[c] = X_test[c].astype(str)

oe = OrdinalEncoder(
    handle_unknown="use_encoded_value",
    unknown_value=-1
)

X[cat_cols] = oe.fit_transform(X[cat_cols])
X_test[cat_cols] = oe.transform(X_test[cat_cols])

print("Features:", feature_cols)

Features: ['product_weight_kg', 'fat_content', 'shelf_visibility', 'product_category', 'product_price', 'store_age_years', 'store_size', 'store_location_tier', 'store_format', 'price_per_kg', 'store_avg_price', 'cat_avg_price', 'price_vs_cat_avg', 'vis_vs_cat_avg', 'store_age_bin']


In [17]:
#tree-ensemble configurations.
rf_cfg = dict(
    n_estimators=300,
    max_depth=8,
    min_samples_leaf=15,
    max_features=0.5
)

hgb_cfg = dict(
    max_iter=300,
    learning_rate=0.05,
    max_depth=3,
    max_leaf_nodes=15,
    min_samples_leaf=15
)

rf = RandomForestRegressor(
    random_state=SEED,
    n_jobs=-1,
    **rf_cfg
)

hgb = HistGradientBoostingRegressor(
    random_state=SEED,
    **hgb_cfg
)

rf.fit(X, y)
hgb.fit(X, y)

tree_ensemble_pred = (
    0.70 * rf.predict(X_test)
    + 0.30 * hgb.predict(X_test)
)

tree_ensemble_pred = np.clip(tree_ensemble_pred, 0, None)

print(pd.Series(tree_ensemble_pred).describe())

count    1705.000000
mean     2190.018540
std      1305.427401
min        56.607915
25%      1185.958843
50%      2077.944797
75%      3061.435505
max      6235.850562
dtype: float64


## 2. Historical product/store baselines

In [18]:
global_sales_mean = train["total_sales"].mean()

product_sales_mean = (
    train.groupby("product_code")["total_sales"]
    .mean()
)

store_sales_mean = (
    train.groupby("store_code")["total_sales"]
    .mean()
)

product_pred = (
    test["product_code"]
    .map(product_sales_mean)
    .fillna(global_sales_mean)
    .to_numpy()
)

store_pred = (
    test["store_code"]
    .map(store_sales_mean)
    .fillna(global_sales_mean)
    .to_numpy()
)

print("Global sales mean:", global_sales_mean)

Global sales mean: 2174.7565737753007


## 3. Reconstruct the `t=3.71` anchor internally

In [19]:
base_t0 = (
    0.96 * tree_ensemble_pred
    + 0.02 * product_pred
    + 0.02 * store_pred
)

base_t1 = (
    0.97 * tree_ensemble_pred
    + 0.015 * product_pred
    + 0.015 * store_pred
)

T = 3.71

anchor_t371 = (
    base_t0
    + T * (base_t1 - base_t0)
)

anchor_t371 = np.clip(anchor_t371, 0, None)

print("t=0 mean :", base_t0.mean())
print("t=1 mean :", base_t1.mean())
print("t=3.71 mean:", anchor_t371.mean())

t=0 mean : 2189.8914762376776
t=1 mean : 2189.923242116216
t=3.71 mean: 2190.0093276470548


## 4. V3 units ensemble

In [20]:
kf = KFold(n_splits=5, shuffle=True, random_state=SEED)

tr = train.copy()
te = test.copy()

y_np = tr["total_sales"].to_numpy()
units = y_np / tr["product_price"].to_numpy()


# Store multiplier baseline

base_o = np.zeros(len(tr))
base_t = np.zeros(len(te))

for a, b in kf.split(tr):
    A = tr.iloc[a]
    B = tr.iloc[b]

    sm = (
        pd.DataFrame({"store": A["store_code"], "units": units[a]})
        .groupby("store")["units"]
        .mean()
    )

    gm = units[a].mean()

    base_o[b] = (
        B["store_code"].map(sm).fillna(gm).to_numpy()
        * B["product_price"].to_numpy()
    )

store_units = (
    tr.assign(units=units)
    .groupby("store_code")["units"]
    .mean()
)

base_t = (
    te["store_code"].map(store_units).fillna(units.mean()).to_numpy()
    * te["product_price"].to_numpy()
)


# Product residual model

prod_o = np.zeros(len(tr))
prod_t = np.zeros(len(te))

for a, b in kf.split(tr):
    A = tr.iloc[a].copy()
    B = tr.iloc[b].copy()

    A["units"] = units[a]

    gm = A["units"].mean()

    sm = A.groupby("store_code")["units"].mean()

    A["residual"] = (
        A["units"]
        - A["store_code"].map(sm).fillna(gm)
    )

    q = A.groupby("product_code")["residual"].agg(["mean", "count"])

    shrink = 50.0

    pe = (
        q["mean"] * q["count"]
        / (q["count"] + shrink)
    )

    pred_units = (
        B["store_code"].map(sm).fillna(gm).to_numpy()
        + B["product_code"].map(pe).fillna(0).to_numpy()
    )

    prod_o[b] = (
        pred_units
        * B["product_price"].to_numpy()
    )

    prod_t += (
        te["store_code"].map(sm).fillna(gm).to_numpy()
        + te["product_code"].map(pe).fillna(0).to_numpy()
    ) * te["product_price"].to_numpy() / 5.0

print("Store/product components ready.")

Store/product components ready.


In [21]:

# V3 CatBoost units model

allx = pd.concat(
    [tr.drop(columns="total_sales"), te],
    ignore_index=True
)

cat = [
    "product_code",
    "product_category",
    "fat_content",
    "store_code",
    "store_size",
    "store_location_tier",
    "store_format",
    "product_dept",
    "format_x_tier",
    "store_cat"
]

for c in [
    "product_category",
    "fat_content",
    "store_code",
    "store_size",
    "store_location_tier",
    "store_format",
    "product_code"
]:
    allx[c] = (
        allx[c]
        .fillna("MISSING")
        .astype(str)
        .str.strip()
        .str.lower()
    )

allx["product_dept"] = allx["product_code"].str[:5]
allx["format_x_tier"] = (
    allx["store_format"] + "__"
    + allx["store_location_tier"]
)
allx["store_cat"] = (
    allx["store_code"] + "__"
    + allx["product_category"]
)

pm = (
    allx.groupby("product_code")["product_weight_kg"]
    .transform("median")
)

cm = (
    allx.groupby("product_category")["product_weight_kg"]
    .transform("median")
)

allx["product_weight_kg"] = (
    allx["product_weight_kg"]
    .fillna(pm)
    .fillna(cm)
    .fillna(allx["product_weight_kg"].median())
)

allx.loc[
    allx["shelf_visibility"].eq(0),
    "shelf_visibility"
] = np.nan

pv = (
    allx.groupby("product_code")["shelf_visibility"]
    .transform("median")
)

sv = (
    allx.groupby("store_code")["shelf_visibility"]
    .transform("median")
)

allx["shelf_visibility"] = (
    allx["shelf_visibility"]
    .fillna(pv)
    .fillna(sv)
    .fillna(allx["shelf_visibility"].median())
)

for c in [
    "product_price",
    "product_weight_kg",
    "shelf_visibility",
    "store_age_years"
]:
    allx[c] = allx[c].fillna(allx[c].median())

allx["price_log"] = np.log1p(allx["product_price"])
allx["weight_log"] = np.log1p(allx["product_weight_kg"])
allx["vis_log"] = np.log1p(allx["shelf_visibility"])

allx["price_density"] = (
    allx["product_price"]
    / allx["product_weight_kg"].clip(lower=0.1)
)

allx["price_to_store"] = (
    allx["product_price"]
    / allx.groupby("store_code")["product_price"].transform("mean")
)

allx["price_to_cat"] = (
    allx["product_price"]
    / allx.groupby("product_category")["product_price"].transform("mean")
)

allx["vis_to_cat"] = (
    allx["shelf_visibility"]
    / allx.groupby("product_category")["shelf_visibility"].transform("mean")
)

for c in [
    "product_code",
    "store_code",
    "product_category",
    "store_cat",
    "product_dept"
]:
    allx[c + "_count"] = allx[c].map(
        allx[c].value_counts()
    )

features = [c for c in allx.columns if c != "id"]

CX = allx.iloc[:len(tr)].copy()
CT = allx.iloc[len(tr):].copy()

for c in cat:
    CX[c] = CX[c].astype(str)
    CT[c] = CT[c].astype(str)

cb_o = np.zeros(len(tr))
cb_t = np.zeros(len(te))

for fold, (a, b) in enumerate(kf.split(CX), 1):

    model = CatBoostRegressor(
        iterations=2400,
        depth=7,
        learning_rate=0.025,
        l2_leaf_reg=10,
        loss_function="RMSE",
        random_seed=42 + fold,
        verbose=False,
        thread_count=-1
    )

    model.fit(
        CX.iloc[a][features],
        units[a],
        cat_features=cat,
        eval_set=(CX.iloc[b][features], units[b]),
        early_stopping_rounds=120,
        verbose=False
    )

    cb_o[b] = (
        model.predict(CX.iloc[b][features])
        * CX.iloc[b]["product_price"].to_numpy()
    )

    cb_t += (
        model.predict(CT[features])
        * CT["product_price"].to_numpy()
        / 5.0
    )

print("CatBoost component ready.")

CatBoost component ready.


In [22]:

# V3 Ridge units model

ridge_o = np.zeros(len(tr))
ridge_t = np.zeros(len(te))

ridge_cats = [
    "product_code",
    "store_code",
    "product_category",
    "store_format",
    "store_location_tier",
    "store_size",
    "fat_content"
]

ridge_nums = [
    "product_price",
    "product_weight_kg",
    "shelf_visibility",
    "store_age_years"
]

R = allx[ridge_cats + ridge_nums].copy()

R[ridge_nums] = R[ridge_nums].fillna(
    R[ridge_nums].median()
)

R[ridge_cats] = R[ridge_cats].fillna("M").astype(str)

for a, b in kf.split(R.iloc[:len(tr)]):

    pre = ColumnTransformer([
        (
            "num",
            StandardScaler(),
            ridge_nums
        ),
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            ridge_cats
        )
    ])

    model = make_pipeline(
        pre,
        Ridge(alpha=50)
    )

    model.fit(
        R.iloc[a],
        units[a]
    )

    ridge_o[b] = (
        model.predict(R.iloc[b])
        * tr.iloc[b]["product_price"].to_numpy()
    )

    ridge_t += (
        model.predict(R.iloc[len(tr):])
        * te["product_price"].to_numpy()
        / 5.0
    )

print("Ridge component ready.")

Ridge component ready.


## 5. Select the V3 ensemble

In [23]:
def rmse(y_true, pred):
    return np.sqrt(
        mean_squared_error(
            y_true,
            np.clip(pred, 0, None)
        )
    )

components = {
    "base": base_o,
    "prod_res": prod_o,
    "catboost": cb_o,
    "ridge": ridge_o
}

for name, pred in components.items():
    print(name, rmse(y_np, pred))

best = (np.inf, None)

for w1 in np.linspace(0, 1, 21):
    for w2 in np.linspace(0, 1 - w1, 21):
        for w3 in np.linspace(0, 1 - w1 - w2, 21):

            w4 = 1 - w1 - w2 - w3

            p = (
                w1 * base_o
                + w2 * prod_o
                + w3 * cb_o
                + w4 * ridge_o
            )

            score = rmse(y_np, p)

            if score < best[0]:
                best = (score, (w1, w2, w3, w4))

print("Coarse best:", best)

bw = best[1]
best2 = best

for w1 in np.linspace(
    max(0, bw[0] - 0.1),
    min(1, bw[0] + 0.1),
    21
):
    for w2 in np.linspace(
        max(0, bw[1] - 0.1),
        min(1 - w1, bw[1] + 0.1),
        21
    ):
        for w3 in np.linspace(
            max(0, bw[2] - 0.1),
            min(1 - w1 - w2, bw[2] + 0.1),
            21
        ):

            w4 = 1 - w1 - w2 - w3

            p = (
                w1 * base_o
                + w2 * prod_o
                + w3 * cb_o
                + w4 * ridge_o
            )

            score = rmse(y_np, p)

            if score < best2[0]:
                best2 = (
                    score,
                    (w1, w2, w3, w4)
                )

print("Refined best:", best2)

v3_weights = best2[1]

base 1071.9101489074196
prod_res 1070.9361010253795
catboost 1071.2523838755556
ridge 1072.6396623959247
Coarse best: (np.float64(1070.486875465111), (np.float64(0.0), np.float64(0.55), np.float64(0.44999999999999996), np.float64(0.0)))
Refined best: (np.float64(1070.4862841859926), (np.float64(0.0), np.float64(0.5700000000000001), np.float64(0.42999999999999994), np.float64(0.0)))


In [24]:

w1, w2, w3, w4 = v3_weights

v3_prediction = (
    w1 * base_t
    + w2 * prod_t
    + w3 * cb_t
    + w4 * ridge_t
)


v3_prediction = np.clip(v3_prediction, 0, None)

print("V3 prediction:")
print(pd.Series(v3_prediction).describe())

V3 prediction:
count    1705.000000
mean     2193.364997
std      1324.905366
min        78.254540
25%      1179.635578
50%      2065.323759
75%      3043.969005
max      6832.867620
dtype: float64


## 6. Final

In [25]:
DIRECTION_STEP = 0.30

final_pred = (
    anchor_t371
    + DIRECTION_STEP
    * (v3_prediction - anchor_t371)
)

final_pred = np.clip(final_pred, 0, None)
final_pred = np.round(final_pred, 2)

submission = pd.DataFrame({
    "id": test["id"],
    "total_sales": final_pred
})

assert len(submission) == len(test)
assert submission["id"].equals(test["id"])
assert submission["total_sales"].notna().all()
assert np.isfinite(submission["total_sales"]).all()
assert (submission["total_sales"] >= 0).all()

submission.to_csv(OUTPUT_PATH, index=False)

print("Saved:", OUTPUT_PATH)
print("Shape:", submission.shape)
print()
print(submission.head(10))
print()
print(submission["total_sales"].describe())

Saved: /content/submission_18.csv
Shape: (1705, 2)

          id  total_sales
0  row_00009      2840.96
1  row_00015      4451.66
2  row_00019      3087.40
3  row_00020      3170.62
4  row_00023      2537.91
5  row_00026       384.15
6  row_00027      3915.53
7  row_00033      3061.33
8  row_00034       355.32
9  row_00037      2025.53

count    1705.000000
mean     2191.016006
std      1308.331341
min        65.660000
25%      1182.670000
50%      2081.880000
75%      3065.520000
max      6352.000000
Name: total_sales, dtype: float64
